<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Load dataset with all columns**

In [ ]:


import duckdb
from huggingface_hub import get_token

# 1. Setup Auth Token & Connection
token = get_token()
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

# 2. Query to Fetch EXACTLY 1 Row
query_single_row = f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 1;
"""

# 3. Execute Query
df_single = con.sql(query_single_row).df()

# 4. Print All 31 Column Names strictly as a Clean List
print("=" * 65)
print("ALL 31 COLUMNS IN RAW DATASET:")
print("=" * 65)

for idx, col_name in enumerate(df_single.columns, 1):
    print(f"{idx:02d}. {col_name}")

print("\n" + "=" * 65)
print("SINGLE ROW DATA SAMPLE:")
print("=" * 65)
display(df_single)




ALL 31 COLUMNS IN RAW DATASET:
01. report_date
02. client_hash_id
03. content_hash_id
04. client_has_gsc
05. client_has_ga4
06. gsc_data_available
07. ga4_data_available
08. gsc_impressions
09. gsc_clicks
10. gsc_sum_position
11. gsc_avg_position
12. ga4_pageviews
13. ga4_sessions
14. ga4_users
15. ga4_engaged_sessions
16. ga4_total_engagement_sec
17. sessions_organic
18. sessions_direct
19. sessions_referral
20. sessions_social
21. sessions_paid
22. sessions_ai
23. ai_chatgpt
24. ai_perplexity
25. ai_gemini
26. ai_copilot
27. ai_claude
28. ai_meta
29. ai_other
30. scroll_events
31. month

SINGLE ROW DATA SAMPLE:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [1]:
import duckdb
from huggingface_hub import get_token

# 1. Setup Auth Token & Connection
token = get_token()
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

# 2. Query to fetch all distinct months using Hive Partitioning
query_months = f"""
SELECT DISTINCT month
FROM read_parquet('{rel}/fact_content_daily_performance/*/*.parquet', hive_partitioning=1)
ORDER BY month ASC;
"""

# 3. Execute Query
print("Fetching month partitions from Hugging Face Warehouse...")
df_months = con.sql(query_months).df()

# 4. Output Results
print("=" * 65)
print(f"TOTAL MONTHS AVAILABLE IN DATASET: {len(df_months)}")
print("=" * 65)
print(df_months.to_string(index=False))

print("\n" + "=" * 65)
print(f"START MONTH : {df_months['month'].iloc[0]}")
print(f"END MONTH   : {df_months['month'].iloc[-1]}")
print("=" * 65)

Fetching month partitions from Hugging Face Warehouse...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TOTAL MONTHS AVAILABLE IN DATASET: 18
  month
2025-01
2025-02
2025-03
2025-04
2025-05
2025-06
2025-07
2025-08
2025-09
2025-10
2025-11
2025-12
2026-01
2026-02
2026-03
2026-04
2026-05
2026-06

START MONTH : 2025-01
END MONTH   : 2026-06


**Load Dataset of month total 18 month**

In [2]:
import duckdb
from huggingface_hub import get_token

# --------------------------------------------------
# 1. Setup
# --------------------------------------------------

token = get_token()

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret
    (
        TYPE HUGGINGFACE,
        TOKEN '{token}'
    );
    """
)

# Fast threading / lower memory pressure
con.execute("SET preserve_insertion_order = false;")
con.execute("SET threads = 4;")

rel = "hf://datasets/FlyRank/internship-warehouse"


# --------------------------------------------------
# 2. Load 18 Months -> Monthly Page-Level Data
# --------------------------------------------------

query_monthly = f"""
WITH daily_data AS
(
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position,
        ga4_total_engagement_sec,
        sessions_organic,
        sessions_ai,
        ai_other

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/*/*.parquet',
        hive_partitioning = true
    )

    WHERE report_date >= DATE '2025-01-01'
      AND report_date <  DATE '2026-07-01'
)

SELECT
    client_hash_id,
    content_hash_id,

    DATE_TRUNC('month', report_date) AS month,

    SUM(gsc_clicks) AS gsc_clicks,
    SUM(gsc_impressions) AS gsc_impressions,

    -- Monthly average of the daily average position
    AVG(gsc_avg_position) AS gsc_avg_position,

    SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,

    SUM(sessions_organic) AS sessions_organic,
    SUM(sessions_ai) AS sessions_ai,
    SUM(ai_other) AS ai_other

FROM daily_data

GROUP BY
    client_hash_id,
    content_hash_id,
    DATE_TRUNC('month', report_date)

ORDER BY
    content_hash_id,
    month;
"""

df_monthly = con.sql(query_monthly).df()


# --------------------------------------------------
# 3. Basic Verification
# --------------------------------------------------

print("=" * 65)
print("MONTHLY PAGE-LEVEL DATA")
print("=" * 65)

print("Rows:", len(df_monthly))
print("Columns:", len(df_monthly.columns))
print("Unique Pages:", df_monthly["content_hash_id"].nunique())
print("Unique Clients:", df_monthly["client_hash_id"].nunique())
print("Unique Months:", df_monthly["month"].nunique())

print("\nMonths:")
print(
    df_monthly["month"]
    .drop_duplicates()
    .sort_values()
    .dt.strftime("%Y-%m")
    .tolist()
)

print("\nFirst 5 rows:")
display(df_monthly.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

MONTHLY PAGE-LEVEL DATA
Rows: 2871202
Columns: 10
Unique Pages: 427292
Unique Clients: 70
Unique Months: 18

Months:
['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']

First 5 rows:


,client_hash_id,content_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,ai_other
0,client_9958f0a7ae1df715,content_000005d4ced12088,2025-03-01,0.0,7.0,9.333333,0.0,0.0,0.0,0.0
1,client_9958f0a7ae1df715,content_000005d4ced12088,2025-04-01,1.0,146.0,35.762918,0.0,0.0,0.0,0.0
2,client_9958f0a7ae1df715,content_000005d4ced12088,2025-05-01,0.0,257.0,38.982641,0.0,0.0,0.0,0.0
3,client_9958f0a7ae1df715,content_000005d4ced12088,2025-06-01,0.0,139.0,37.522978,0.0,0.0,0.0,0.0
4,client_9958f0a7ae1df715,content_000005d4ced12088,2025-07-01,0.0,254.0,35.843110,0.0,0.0,0.0,0.0


**Audit Missing Value**

In [10]:
# ============================================
# STEP: Missing Value Audit
# ============================================

import pandas as pd

# Original dataset safe rahega
df_copy = df_monthly.copy()

print("=" * 70)
print("MISSING VALUE AUDIT")
print("=" * 70)

# Total rows
print(f"Total rows: {len(df_copy):,}")

# Missing count
missing_count = df_copy.isna().sum()

# Missing percentage
missing_percent = (
    missing_count / len(df_copy) * 100
).round(2)

# Create audit table
missing_report = pd.DataFrame({
    "column": df_copy.columns,
    "missing_count": missing_count.values,
    "missing_percent": missing_percent.values
})

# Sirf missing wali columns
missing_report = (
    missing_report[
        missing_report["missing_count"] > 0
    ]
    .sort_values("missing_count", ascending=False)
    .reset_index(drop=True)
)

print("\nColumns containing missing values:")
display(missing_report)

MISSING VALUE AUDIT
Total rows: 2,871,202

Columns containing missing values:


,column,missing_count,missing_percent
0,gsc_avg_position,1324747,46.14
1,ga4_total_engagement_sec,950756,33.11
2,sessions_organic,950756,33.11
3,sessions_ai,950756,33.11
4,ai_other,950756,33.11


**Check Missing value pattern**

In [15]:
# ============================================
# MISSING VALUE GROUP ANALYSIS
# ============================================

check_cols = [
    "gsc_avg_position",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "ai_other"
]

print("=" * 70)
print("MISSING VALUE GROUP ANALYSIS")
print("=" * 70)

# ------------------------------------------------
# 1. Rows where ALL 4 GA4/AI metrics are missing
# ------------------------------------------------

ga4_ai_cols = [
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "ai_other"
]

all_ga4_ai_missing = df_copy[ga4_ai_cols].isna().all(axis=1)

print("\n1. ALL GA4/AI columns missing together:")
print(f"Rows: {all_ga4_ai_missing.sum():,}")
print(f"Percentage: {all_ga4_ai_missing.mean() * 100:.2f}%")


# ------------------------------------------------
# 2. Rows where GA4/AI columns are partially missing
# ------------------------------------------------

partial_ga4_ai = (
    df_copy[ga4_ai_cols].isna().any(axis=1)
    &
    ~df_copy[ga4_ai_cols].isna().all(axis=1)
)

print("\n2. PARTIAL GA4/AI missing:")
print(f"Rows: {partial_ga4_ai.sum():,}")
print(f"Percentage: {partial_ga4_ai.mean() * 100:.2f}%")


# ------------------------------------------------
# 3. GSC average position missing
# ------------------------------------------------

gsc_missing = df_copy["gsc_avg_position"].isna()

print("\n3. GSC average position missing:")
print(f"Rows: {gsc_missing.sum():,}")
print(f"Percentage: {gsc_missing.mean() * 100:.2f}%")


# ------------------------------------------------
# 4. Combined pattern
# ------------------------------------------------

pattern = pd.DataFrame({
    "gsc_avg_position_missing": df_copy["gsc_avg_position"].isna(),
    "ga4_engagement_missing": df_copy["ga4_total_engagement_sec"].isna(),
    "organic_missing": df_copy["sessions_organic"].isna(),
    "ai_sessions_missing": df_copy["sessions_ai"].isna(),
    "ai_other_missing": df_copy["ai_other"].isna()
})

print("\n4. Most common missing-value patterns:")
display(
    pattern.value_counts()
    .head(10)
    .reset_index(name="row_count")
)

MISSING VALUE GROUP ANALYSIS

1. ALL GA4/AI columns missing together:
Rows: 950,756
Percentage: 33.11%

2. PARTIAL GA4/AI missing:
Rows: 0
Percentage: 0.00%

3. GSC average position missing:
Rows: 1,324,747
Percentage: 46.14%

4. Most common missing-value patterns:


,gsc_avg_position_missing,ga4_engagement_missing,organic_missing,ai_sessions_missing,ai_other_missing,row_count
0,False,False,False,False,False,1097142
1,True,False,False,False,False,823304
2,True,True,True,True,True,501443
3,False,True,True,True,True,449313


In [16]:
# ============================================
# STEP: Missing vs Non-Missing Value Comparison
# ============================================

print("=" * 70)
print("MISSING vs NON-MISSING VALUE COMPARISON")
print("=" * 70)

# GA4/AI columns
ga4_ai_cols = [
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "ai_other"
]

# Rows where complete GA4/AI block is available
ga4_available = df_copy[ga4_ai_cols].notna().all(axis=1)

print("\nGA4/AI available rows:")
print(f"{ga4_available.sum():,}")

print("\nGA4/AI unavailable rows:")
print(f"{(~ga4_available).sum():,}")


# Compare GSC clicks and impressions
comparison_cols = [
    "gsc_clicks",
    "gsc_impressions"
]

print("\n" + "=" * 70)
print("GSC METRICS")
print("=" * 70)

for col in comparison_cols:

    print(f"\n{col}")

    print("When GA4/AI is available:")
    print(
        df_copy.loc[ga4_available, col]
        .describe()[["count", "mean", "50%", "min", "max"]]
    )

    print("\nWhen GA4/AI is unavailable:")
    print(
        df_copy.loc[~ga4_available, col]
        .describe()[["count", "mean", "50%", "min", "max"]]
    )

MISSING vs NON-MISSING VALUE COMPARISON

GA4/AI available rows:
1,920,446

GA4/AI unavailable rows:
950,756

GSC METRICS

gsc_clicks
When GA4/AI is available:
count    1.920446e+06
mean     2.610195e+00
50%      0.000000e+00
min      0.000000e+00
max      1.521700e+05
Name: gsc_clicks, dtype: float64

When GA4/AI is unavailable:
count    950756.000000
mean          1.502067
50%           0.000000
min           0.000000
max        4929.000000
Name: gsc_clicks, dtype: float64

gsc_impressions
When GA4/AI is available:
count    1.920446e+06
mean     6.893800e+02
50%      3.000000e+00
min      0.000000e+00
max      7.993580e+05
Name: gsc_impressions, dtype: float64

When GA4/AI is unavailable:
count    950756.000000
mean        462.371879
50%           0.000000
min           0.000000
max      353426.000000
Name: gsc_impressions, dtype: float64


In [17]:
# ============================================
# STEP: GSC Position Missing Analysis
# ============================================

position_missing = df_copy["gsc_avg_position"].isna()
position_available = df_copy["gsc_avg_position"].notna()

print("=" * 70)
print("GSC AVG POSITION: MISSING vs AVAILABLE")
print("=" * 70)

print(f"\nPosition available : {position_available.sum():,}")
print(f"Position missing   : {position_missing.sum():,}")


for col in ["gsc_clicks", "gsc_impressions"]:

    print("\n" + "-" * 60)
    print(col)

    print("\nWhen position is AVAILABLE:")
    print(
        df_copy.loc[position_available, col]
        .describe()[["count", "mean", "50%", "min", "max"]]
    )

    print("\nWhen position is MISSING:")
    print(
        df_copy.loc[position_missing, col]
        .describe()[["count", "mean", "50%", "min", "max"]]
    )

GSC AVG POSITION: MISSING vs AVAILABLE

Position available : 1,546,455
Position missing   : 1,324,747

------------------------------------------------------------
gsc_clicks

When position is AVAILABLE:
count    1.546455e+06
mean     4.164904e+00
50%      0.000000e+00
min      0.000000e+00
max      1.521700e+05
Name: gsc_clicks, dtype: float64

When position is MISSING:
count    1.324747e+06
mean     7.548611e-07
50%      0.000000e+00
min      0.000000e+00
max      1.000000e+00
Name: gsc_clicks, dtype: float64

------------------------------------------------------------
gsc_impressions

When position is AVAILABLE:
count    1.546455e+06
mean     1.140363e+03
50%      1.070000e+02
min      1.000000e+00
max      7.993580e+05
Name: gsc_impressions, dtype: float64

When position is MISSING:
count    1.324747e+06
mean     2.113611e-05
50%      0.000000e+00
min      0.000000e+00
max      2.800000e+01
Name: gsc_impressions, dtype: float64


In [19]:
# ============================================
# STEP: GA4 / AI Missing Block Analysis
# ============================================

ga4_ai_cols = [
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "ai_other"
]

# Rows where all GA4/AI metrics are missing
ga4_missing = df_copy[ga4_ai_cols].isna().all(axis=1)

print("=" * 70)
print("GA4 / AI MISSING BLOCK ANALYSIS")
print("=" * 70)

print(f"\nGA4/AI missing rows: {ga4_missing.sum():,}")
print(f"GA4/AI available rows: {(~ga4_missing).sum():,}")

# ------------------------------------------------
# 1. GSC activity in GA4/AI-missing rows
# ------------------------------------------------

print("\n" + "=" * 70)
print("GSC ACTIVITY WHEN GA4/AI IS MISSING")
print("=" * 70)

for col in ["gsc_clicks", "gsc_impressions"]:

    print(f"\n{col}")

    print(
        df_copy.loc[ga4_missing, col]
        .describe()[["count", "mean", "50%", "min", "max"]]
    )


# ------------------------------------------------
# 2. Monthly distribution of missing GA4/AI
# ------------------------------------------------

print("\n" + "=" * 70)
print("GA4/AI MISSING BY MONTH")
print("=" * 70)

monthly_missing = (
    df_copy.loc[ga4_missing]
    .groupby("month")
    .size()
    .reset_index(name="missing_rows")
)

display(monthly_missing)


# ------------------------------------------------
# 3. Percentage of each month's rows affected
# ------------------------------------------------

monthly_total = (
    df_copy
    .groupby("month")
    .size()
    .reset_index(name="total_rows")
)

monthly_check = monthly_total.merge(
    monthly_missing,
    on="month",
    how="left"
)

monthly_check["missing_rows"] = (
    monthly_check["missing_rows"].fillna(0)
)

monthly_check["missing_percent"] = (
    monthly_check["missing_rows"]
    / monthly_check["total_rows"]
    * 100
).round(2)

print("\n" + "=" * 70)
print("MONTHLY GA4/AI MISSING PERCENTAGE")
print("=" * 70)

display(monthly_check)

GA4 / AI MISSING BLOCK ANALYSIS

GA4/AI missing rows: 950,756
GA4/AI available rows: 1,920,446

GSC ACTIVITY WHEN GA4/AI IS MISSING

gsc_clicks
count    950756.000000
mean          1.502067
50%           0.000000
min           0.000000
max        4929.000000
Name: gsc_clicks, dtype: float64

gsc_impressions
count    950756.000000
mean        462.371879
50%           0.000000
min           0.000000
max      353426.000000
Name: gsc_impressions, dtype: float64

GA4/AI MISSING BY MONTH


,month,missing_rows
0,2025-11-01,146879
1,2025-12-01,175668
2,2026-01-01,171756
3,2026-02-01,149677
4,2026-03-01,70700
5,2026-04-01,76352
6,2026-05-01,78969
7,2026-06-01,80755



MONTHLY GA4/AI MISSING PERCENTAGE


,month,total_rows,missing_rows,missing_percent
0,2025-01-01,476,0.0,0.00
1,2025-02-01,5903,0.0,0.00
2,2025-03-01,10374,0.0,0.00
3,2025-04-01,13046,0.0,0.00
4,2025-05-01,14887,0.0,0.00
5,2025-06-01,16399,0.0,0.00
6,2025-07-01,27945,0.0,0.00
7,2025-08-01,37204,0.0,0.00
8,2025-09-01,53127,0.0,0.00
9,2025-10-01,110339,0.0,0.00


In [20]:
# ============================================
# STEP: Client-Level GA4/AI Availability
# ============================================

ga4_ai_cols = [
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "ai_other"
]

ga4_missing = df_copy[ga4_ai_cols].isna().all(axis=1)

client_check = (
    df_copy.assign(ga4_ai_missing=ga4_missing)
    .groupby("client_hash_id")
    .agg(
        total_rows=("client_hash_id", "size"),
        missing_rows=("ga4_ai_missing", "sum")
    )
    .reset_index()
)

client_check["missing_percent"] = (
    client_check["missing_rows"]
    / client_check["total_rows"]
    * 100
).round(2)

print("=" * 70)
print("CLIENT-LEVEL GA4/AI MISSINGNESS")
print("=" * 70)

display(
    client_check
    .sort_values("missing_percent", ascending=False)
    .head(20)
)

CLIENT-LEVEL GA4/AI MISSINGNESS


,client_hash_id,total_rows,missing_rows,missing_percent
2,client_0797ff3a1fc9a6a5,2080,2080,100.00
32,client_770e8e5faa9cddfe,1600,1600,100.00
17,client_2b4306c3ed003f01,60630,60630,100.00
18,client_2c32078d69f2cbad,92,92,100.00
19,client_2e65897d94f60220,20168,20168,100.00
12,client_1d09b519bdde7c7a,7128,7128,100.00
43,client_8ddc46da5414ffd8,12271,12271,100.00
36,client_80ee5b7bd5f4eb89,1568,1568,100.00
39,client_861cdcccf8049915,28544,28544,100.00
38,client_835f9123c933bc01,16024,16024,100.00


In [21]:
# ============================================
# STEP: Client + Month Missingness Check
# ============================================

ga4_ai_cols = [
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "ai_other"
]

df_check = df_copy.copy()

df_check["ga4_ai_missing"] = (
    df_check[ga4_ai_cols]
    .isna()
    .all(axis=1)
)

client_month_check = (
    df_check
    .groupby(["client_hash_id", "month"])
    .agg(
        total_rows=("client_hash_id", "size"),
        missing_rows=("ga4_ai_missing", "sum")
    )
    .reset_index()
)

client_month_check["missing_percent"] = (
    client_month_check["missing_rows"]
    / client_month_check["total_rows"]
    * 100
).round(2)

print("=" * 70)
print("CLIENT + MONTH GA4/AI MISSINGNESS")
print("=" * 70)

display(
    client_month_check
    .sort_values(
        ["missing_percent", "total_rows"],
        ascending=[False, False]
    )
    .head(30)
)

CLIENT + MONTH GA4/AI MISSINGNESS


,client_hash_id,month,total_rows,missing_rows,missing_percent
195,client_625b6439094e23e4,2025-11-01,31887,31887,100.0
196,client_625b6439094e23e4,2025-12-01,31887,31887,100.0
197,client_625b6439094e23e4,2026-01-01,31887,31887,100.0
150,client_3ffa76342f366962,2026-01-01,29880,29880,100.0
21,client_08a6a72ff48e62c0,2026-05-01,29812,29812,100.0
22,client_08a6a72ff48e62c0,2026-06-01,29812,29812,100.0
20,client_08a6a72ff48e62c0,2026-04-01,29597,29597,100.0
148,client_3ffa76342f366962,2025-11-01,29578,29577,100.0
149,client_3ffa76342f366962,2025-12-01,29578,29578,100.0
19,client_08a6a72ff48e62c0,2026-03-01,28278,28278,100.0


In [22]:
# ============================================
# STEP: GSC Average Position Missingness
# Client + Month Analysis
# ============================================

df_check = df_copy.copy()

# Missing indicator
df_check["position_missing"] = df_check["gsc_avg_position"].isna()

# Client + Month level analysis
position_check = (
    df_check
    .groupby(["client_hash_id", "month"])
    .agg(
        total_rows=("client_hash_id", "size"),
        missing_rows=("position_missing", "sum"),
        total_clicks=("gsc_clicks", "sum"),
        total_impressions=("gsc_impressions", "sum")
    )
    .reset_index()
)

position_check["missing_percent"] = (
    position_check["missing_rows"]
    / position_check["total_rows"]
    * 100
).round(2)

print("=" * 70)
print("GSC AVG POSITION — CLIENT + MONTH MISSINGNESS")
print("=" * 70)

display(
    position_check
    .sort_values(
        ["missing_percent", "total_rows"],
        ascending=[False, False]
    )
    .head(30)
)

GSC AVG POSITION — CLIENT + MONTH MISSINGNESS


,client_hash_id,month,total_rows,missing_rows,total_clicks,total_impressions,missing_percent
195,client_625b6439094e23e4,2025-11-01,31887,31886,0.0,3.0,100.0
196,client_625b6439094e23e4,2025-12-01,31887,31887,0.0,0.0,100.0
197,client_625b6439094e23e4,2026-01-01,31887,31887,0.0,0.0,100.0
198,client_625b6439094e23e4,2026-02-01,31887,31887,0.0,0.0,100.0
199,client_625b6439094e23e4,2026-03-01,31887,31887,0.0,0.0,100.0
200,client_625b6439094e23e4,2026-04-01,31887,31887,0.0,0.0,100.0
201,client_625b6439094e23e4,2026-05-01,31887,31887,0.0,0.0,100.0
202,client_625b6439094e23e4,2026-06-01,31887,31887,0.0,0.0,100.0
480,client_e547b89c05043229,2025-10-01,9308,9308,0.0,0.0,100.0
62,client_19b89ee4fe3db6da,2026-02-01,6871,6871,0.0,0.0,100.0


In [23]:
# ============================================
# STEP: Create Missing Indicators
# ============================================

# Work on copy
df_clean = df_copy.copy()

# --------------------------------------------
# 1. GSC position missing indicator
# --------------------------------------------

df_clean["gsc_avg_position_missing"] = (
    df_clean["gsc_avg_position"].isna().astype(int)
)

# GSC position: NaN -> 0
df_clean["gsc_avg_position"] = (
    df_clean["gsc_avg_position"].fillna(0)
)


# --------------------------------------------
# 2. GA4 / AI missing indicators
# --------------------------------------------

ga4_ai_cols = [
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "ai_other"
]

for col in ga4_ai_cols:
    df_clean[f"{col}_missing"] = (
        df_clean[col].isna().astype(int)
    )


# --------------------------------------------
# 3. Verify
# --------------------------------------------

print("=" * 70)
print("MISSING INDICATORS CREATED")
print("=" * 70)

indicator_cols = [
    "gsc_avg_position_missing",
    "ga4_total_engagement_sec_missing",
    "sessions_organic_missing",
    "sessions_ai_missing",
    "ai_other_missing"
]

display(df_clean[indicator_cols].sum().to_frame("missing_rows"))

print("\nRemaining NaN values:")
display(
    df_clean.isna().sum()
    .loc[lambda x: x > 0]
    .to_frame("remaining_nan")
)

print("\nNew shape:", df_clean.shape)

MISSING INDICATORS CREATED


,missing_rows
gsc_avg_position_missing,1324747
ga4_total_engagement_sec_missing,950756
sessions_organic_missing,950756
sessions_ai_missing,950756
ai_other_missing,950756



Remaining NaN values:


,remaining_nan
ga4_total_engagement_sec,950756
sessions_organic,950756
sessions_ai,950756
ai_other,950756



New shape: (2871202, 16)


In [24]:
# ============================================
# STEP: GA4 / AI Imputation Audit
# ============================================

ga4_ai_cols = [
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "ai_other"
]

print("=" * 70)
print("GA4 / AI COLUMNS — IMPUTATION AUDIT")
print("=" * 70)

for col in ga4_ai_cols:

    print(f"\n{'-' * 60}")
    print(f"Column: {col}")

    print(
        df_clean[col]
        .dropna()
        .describe()[
            ["count", "mean", "50%", "min", "max"]
        ]
    )

    zero_count = (df_clean[col].dropna() == 0).sum()
    non_zero_count = (df_clean[col].dropna() > 0).sum()

    print(f"\nExisting zero values     : {zero_count:,}")
    print(f"Existing non-zero values: {non_zero_count:,}")

GA4 / AI COLUMNS — IMPUTATION AUDIT

------------------------------------------------------------
Column: ga4_total_engagement_sec
count    1.920446e+06
mean     2.065048e+01
50%      0.000000e+00
min      0.000000e+00
max      2.048460e+05
Name: ga4_total_engagement_sec, dtype: float64

Existing zero values     : 1,694,093
Existing non-zero values: 226,353

------------------------------------------------------------
Column: sessions_organic
count    1.920446e+06
mean     2.714680e+00
50%      0.000000e+00
min      0.000000e+00
max      1.032970e+05
Name: sessions_organic, dtype: float64

Existing zero values     : 1,650,016
Existing non-zero values: 270,430

------------------------------------------------------------
Column: sessions_ai
count    1.920446e+06
mean     4.354093e-02
50%      0.000000e+00
min      0.000000e+00
max      8.770000e+02
Name: sessions_ai, dtype: float64

Existing zero values     : 1,900,670
Existing non-zero values: 19,776

----------------------------------

**Handle Missing Value**

In [25]:
# ============================================
# STEP: Fill Remaining Missing Values
# ============================================

fill_cols = [
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "ai_other"
]

for col in fill_cols:
    df_clean[col] = df_clean[col].fillna(0)

print("=" * 70)
print("MISSING VALUES FILLED")
print("=" * 70)

remaining_nan = df_clean.isna().sum()
remaining_nan = remaining_nan[remaining_nan > 0]

if len(remaining_nan) == 0:
    print("No missing values remaining.")
else:
    display(remaining_nan.to_frame("remaining_nan"))

print("\nFinal shape:", df_clean.shape)

MISSING VALUES FILLED
No missing values remaining.

Final shape: (2871202, 16)


In [26]:
df_clean.tail(20)

,client_hash_id,content_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,ai_other,missing_count,gsc_avg_position_missing,ga4_total_engagement_sec_missing,sessions_organic_missing,sessions_ai_missing,ai_other_missing
2871182,client_625b6439094e23e4,content_ffffe701567e982c,2026-03-01,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1,1,0,0,0,0
2871183,client_625b6439094e23e4,content_ffffe701567e982c,2026-04-01,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1,1,0,0,0,0
2871184,client_625b6439094e23e4,content_ffffe701567e982c,2026-05-01,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1,1,0,0,0,0
2871185,client_625b6439094e23e4,content_ffffe701567e982c,2026-06-01,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1,1,0,0,0,0
2871186,client_73cda7b4e4f265ea,content_fffff09da8a25da6,2025-03-01,0.0,19.0,51.319728,0.0,0.0,0.0,0.0,0,0,0,0,0,0
2871187,client_73cda7b4e4f265ea,content_fffff09da8a25da6,2025-04-01,0.0,195.0,47.522238,0.0,0.0,0.0,0.0,0,0,0,0,0,0
2871188,client_73cda7b4e4f265ea,content_fffff09da8a25da6,2025-05-01,2.0,628.0,46.624313,0.0,0.0,0.0,0.0,0,0,0,0,0,0
2871189,client_73cda7b4e4f265ea,content_fffff09da8a25da6,2025-06-01,0.0,304.0,71.778984,0.0,0.0,0.0,0.0,0,0,0,0,0,0
2871190,client_73cda7b4e4f265ea,content_fffff09da8a25da6,2025-07-01,0.0,69.0,70.676754,0.0,0.0,0.0,0.0,0,0,0,0,0,0
2871191,client_73cda7b4e4f265ea,content_fffff09da8a25da6,2025-08-01,0.0,91.0,68.983848,0.0,0.0,0.0,0.0,0,0,0,0,0,0


**Filter Inactive Rows**

In [27]:
# ============================================
# STEP: Dead Row Audit
# ============================================

# Work on cleaned dataset
df_check = df_clean.copy()

print("=" * 70)
print("DEAD ROW AUDIT")
print("=" * 70)

print(f"Total rows: {len(df_check):,}")

# ------------------------------------------------
# 1. Completely empty rows
# ------------------------------------------------

all_null_rows = df_check.isna().all(axis=1)

print(f"\nCompletely empty rows: {all_null_rows.sum():,}")


# ------------------------------------------------
# 2. Rows where all measurable numeric metrics = 0
# ------------------------------------------------

metric_cols = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "ai_other"
]

all_zero_metrics = (df_check[metric_cols] == 0).all(axis=1)

print(f"Rows with ALL metrics = 0: {all_zero_metrics.sum():,}")


# ------------------------------------------------
# 3. Rows with no measurable activity
# ------------------------------------------------

dead_rows = all_zero_metrics

print(f"\nPotential dead rows: {dead_rows.sum():,}")
print(
    f"Potential dead row percentage: "
    f"{dead_rows.mean() * 100:.2f}%"
)


# ------------------------------------------------
# 4. Show examples
# ------------------------------------------------

print("\nSample potential dead rows:")

display(
    df_check.loc[
        dead_rows,
        [
            "content_hash_id",
            "client_hash_id",
            "month"
        ] + metric_cols
    ].head(20)
)

DEAD ROW AUDIT
Total rows: 2,871,202

Completely empty rows: 0
Rows with ALL metrics = 0: 1,289,758

Potential dead rows: 1,289,758
Potential dead row percentage: 44.92%

Sample potential dead rows:


,content_hash_id,client_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,ai_other
18,content_00001e488b74b799,client_625b6439094e23e4,2025-11-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19,content_00001e488b74b799,client_625b6439094e23e4,2025-12-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20,content_00001e488b74b799,client_625b6439094e23e4,2026-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0
21,content_00001e488b74b799,client_625b6439094e23e4,2026-02-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22,content_00001e488b74b799,client_625b6439094e23e4,2026-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23,content_00001e488b74b799,client_625b6439094e23e4,2026-04-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0
24,content_00001e488b74b799,client_625b6439094e23e4,2026-05-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25,content_00001e488b74b799,client_625b6439094e23e4,2026-06-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0
42,content_00008950670cb6b5,client_def0955f7a377868,2026-02-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0
44,content_00008950670cb6b5,client_def0955f7a377868,2026-04-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**Get Engineered Features**

In [28]:
# ============================================
# STEP: ENGINEERED FEATURES
# ============================================

import numpy as np
import pandas as pd

# Preserve current dataframe
df_features = df_clean.copy()

# --------------------------------------------
# 1. CTR
# Clicks / Impressions
# --------------------------------------------
df_features["ctr"] = np.where(
    df_features["gsc_impressions"] > 0,
    df_features["gsc_clicks"] / df_features["gsc_impressions"],
    0
)

# --------------------------------------------
# 2. Seconds per Click
# Engagement seconds / Clicks
# --------------------------------------------
df_features["sec_per_click"] = np.where(
    df_features["gsc_clicks"] > 0,
    df_features["ga4_total_engagement_sec"] / df_features["gsc_clicks"],
    0
)

# --------------------------------------------
# 3. AI Share
# AI sessions / (Organic + AI sessions)
# --------------------------------------------
total_relevant_sessions = (
    df_features["sessions_organic"]
    + df_features["sessions_ai"]
)

df_features["ai_share"] = np.where(
    total_relevant_sessions > 0,
    df_features["sessions_ai"] / total_relevant_sessions,
    0
)

# --------------------------------------------
# 4. Engagement per Organic Session
# --------------------------------------------
df_features["engagement_per_organic_session"] = np.where(
    df_features["sessions_organic"] > 0,
    df_features["ga4_total_engagement_sec"]
    / df_features["sessions_organic"],
    0
)

# ============================================
# VERIFY
# ============================================

derived_cols = [
    "ctr",
    "sec_per_click",
    "ai_share",
    "engagement_per_organic_session"
]

print("=" * 70)
print("ENGINEERED FEATURES CREATED")
print("=" * 70)

print("\nDerived columns:")
for col in derived_cols:
    print("-", col)

print("\nOriginal shape :", df_clean.shape)
print("New shape      :", df_features.shape)

display(
    df_features[
        [
            "content_hash_id",
            "client_hash_id",
            "month",
            "gsc_clicks",
            "gsc_impressions",
            "gsc_avg_position",
            "ga4_total_engagement_sec",
            "sessions_organic",
            "sessions_ai"
        ] + derived_cols
    ].head(10)
)

ENGINEERED FEATURES CREATED

Derived columns:
- ctr
- sec_per_click
- ai_share
- engagement_per_organic_session

Original shape : (2871202, 16)
New shape      : (2871202, 20)


,content_hash_id,client_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,ctr,sec_per_click,ai_share,engagement_per_organic_session
0,content_000005d4ced12088,client_9958f0a7ae1df715,2025-03-01,0.0,7.0,9.333333,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
1,content_000005d4ced12088,client_9958f0a7ae1df715,2025-04-01,1.0,146.0,35.762918,0.0,0.0,0.0,0.006849,0.0,0.0,0.0
2,content_000005d4ced12088,client_9958f0a7ae1df715,2025-05-01,0.0,257.0,38.982641,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
3,content_000005d4ced12088,client_9958f0a7ae1df715,2025-06-01,0.0,139.0,37.522978,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
4,content_000005d4ced12088,client_9958f0a7ae1df715,2025-07-01,0.0,254.0,35.843110,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
5,content_000005d4ced12088,client_9958f0a7ae1df715,2025-08-01,1.0,742.0,39.317181,0.0,0.0,0.0,0.001348,0.0,0.0,0.0
6,content_000005d4ced12088,client_9958f0a7ae1df715,2025-09-01,1.0,524.0,37.794280,0.0,0.0,0.0,0.001908,0.0,0.0,0.0
7,content_000005d4ced12088,client_9958f0a7ae1df715,2025-10-01,0.0,197.0,36.865622,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
8,content_000005d4ced12088,client_9958f0a7ae1df715,2025-11-01,0.0,120.0,38.651264,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
9,content_000005d4ced12088,client_9958f0a7ae1df715,2025-12-01,0.0,184.0,38.919439,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


**No Categorical Handling at this stage**

In [29]:
# ============================================
# STEP: CATEGORICAL / STRING AUDIT
# ============================================

# Work on a copy
df_encoded = df_features.copy()

print("=" * 70)
print("CATEGORICAL / STRING COLUMN AUDIT")
print("=" * 70)

# Find object/string columns
categorical_cols = df_encoded.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("\nString / categorical columns:")
for col in categorical_cols:
    print(f"- {col}")

# Show unique values for each
print("\nUnique-value summary:")

for col in categorical_cols:
    print(f"\n{col}")
    print("Unique values:", df_encoded[col].nunique())
    print(df_encoded[col].dropna().unique()[:20])


# ============================================
# MONTH HANDLING
# ============================================

if "month" in df_encoded.columns:

    # Convert month to datetime
    df_encoded["month"] = pd.to_datetime(
        df_encoded["month"],
        errors="coerce"
    )

    # Extract temporal information
    df_encoded["year"] = df_encoded["month"].dt.year
    df_encoded["month_number"] = df_encoded["month"].dt.month

print("\n" + "=" * 70)
print("RESULT")
print("=" * 70)

print("Shape:", df_encoded.shape)

print("\nData types:")
display(
    df_encoded[
        ["content_hash_id", "client_hash_id",
         "month", "year", "month_number"]
    ].dtypes.to_frame("dtype")
)

CATEGORICAL / STRING COLUMN AUDIT

String / categorical columns:
- client_hash_id
- content_hash_id

Unique-value summary:

client_hash_id
Unique values: 70
['client_9958f0a7ae1df715' 'client_7de9989c909e91a5'
 'client_625b6439094e23e4' 'client_73cda7b4e4f265ea'
 'client_def0955f7a377868' 'client_3ffa76342f366962'
 'client_2094c6eb080311d5' 'client_2b4306c3ed003f01'
 'client_65de48885f4ef01b' 'client_08a6a72ff48e62c0'
 'client_62f4a7e64f5e0096' 'client_2910fd937f0b4d9a'
 'client_e5c2aa26a8598242' 'client_3f0ce4d44fe94f3d'
 'client_a60a11451483af1c' 'client_835f9123c933bc01'
 'client_fef1a8f436438636' 'client_ba65e80a1116ae41'
 'client_157ffe4d4a595515' 'client_400c21c81c8b46ef']

content_hash_id
Unique values: 427292
['content_000005d4ced12088' 'content_00000c99413ae2ad'
 'content_00001e488b74b799' 'content_00007bd2985b77c3'
 'content_00008950670cb6b5' 'content_0000a348850eb1fc'
 'content_0000c57e204651e5' 'content_0000cd28fbda69f3'
 'content_0000d31f3926ea12' 'content_0000d495bfbfb4a8

,dtype
content_hash_id,object
client_hash_id,object
month,datetime64[us]
year,int32
month_number,int32


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 2. Feature Notes

| # | Column Name | Type | Meaning | Missing Value Handling | Available Before Prediction? |
|:---|:---|:---|:---|:---|:---|
| 1 | `content_hash_id` | ID | Unique identifier for the web page/content | Preserve; audit if missing | Yes — identifier |
| 2 | `client_hash_id` | ID | Unique identifier for the client/website | Preserve; audit if missing | Yes — identifier |
| 3 | `month` | Date | Month represented by the record | Audit if missing | Yes — historical/current completed month |
| 4 | `gsc_clicks` | Numeric | Number of clicks received from Google Search | Missing → 0 + missing indicator | Yes — if GSC data is available |
| 5 | `gsc_impressions` | Numeric | Number of times the page appeared in Google Search results | Missing → 0 + missing indicator | Yes — if GSC data is available |
| 6 | `gsc_avg_position` | Numeric | Average position of the page in Google Search results | Missing → 0 + missing indicator | Yes — if GSC data is available |
| 7 | `ga4_total_engagement_sec` | Numeric | Total user engagement time in seconds | Missing → 0 + missing indicator | Depends on GA4 data availability |
| 8 | `sessions_organic` | Numeric | Sessions originating from organic search | Missing → 0 + missing indicator | Depends on GA4 data availability |
| 9 | `sessions_ai` | Numeric | Sessions originating from AI sources | Missing → 0 + missing indicator | Depends on AI/GA4 data availability |
| 10 | `ai_other` | Numeric | Sessions from other AI sources not separately classified | Missing → 0 + missing indicator | Depends on AI/GA4 data availability |
| 11 | `gsc_avg_position_missing` | Binary | Indicates whether GSC average position was originally missing | No missing value | Yes — derived from available historical data |
| 12 | `ga4_total_engagement_sec_missing` | Binary | Indicates whether GA4 engagement data was originally missing | No missing value | Depends on GA4 data availability |
| 13 | `sessions_organic_missing` | Binary | Indicates whether organic session data was originally missing | No missing value | Depends on GA4 data availability |
| 14 | `sessions_ai_missing` | Binary | Indicates whether AI session data was originally missing | No missing value | Depends on AI/GA4 data availability |
| 15 | `ai_other_missing` | Binary | Indicates whether `ai_other` was originally missing | No missing value | Depends on AI/GA4 data availability |
| 16 | `ctr` | Derived | Click-through rate calculated from clicks and impressions | If impressions = 0, set to 0 | Depends on GSC data availability |
| 17 | `sec_per_click` | Derived | Average engagement seconds per click | If clicks = 0, set to 0 | Depends on GSC + GA4 data availability |
| 18 | `ai_share` | Derived | Proportion of traffic attributed to AI sources | If denominator = 0, set to 0 | Depends on AI/GA4 data availability |
| 19 | `engagement_per_organic_session` | Derived | Average engagement time per organic session | If organic sessions = 0, set to 0 | Depends on GA4 data availability |
| 20 | `year` / `month_number` | Derived | Year and numeric month extracted from `month` | Derived from month | Yes — if month is available |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**SHOW 3 TEST**

In [30]:
import pandas as pd
import numpy as np

# ============================================================
# STEP 3: LEAKAGE HUNT
# ============================================================

df_leakage = df_features.copy()

print("=" * 70)
print("LEAKAGE HUNT")
print("=" * 70)

print(f"Rows    : {len(df_leakage):,}")
print(f"Columns : {len(df_leakage.columns)}")

# ============================================================
# ATTACK 1: LABEL-DERIVED / FUTURE-LOOKING COLUMN NAMES
# ============================================================

print("\n" + "=" * 70)
print("1. LABEL-DERIVED / FUTURE-LOOKING COLUMN ATTACK")
print("=" * 70)

suspicious_keywords = [
    "target",
    "label",
    "trend",
    "future",
    "next",
    "future_",
    "next_",
    "lead",
    "outcome",
    "prediction",
    "predicted"
]

suspicious_columns = []

for col in df_leakage.columns:
    col_lower = col.lower()

    if any(keyword in col_lower for keyword in suspicious_keywords):
        suspicious_columns.append(col)

if suspicious_columns:
    print("Potentially suspicious columns found:")
    for col in suspicious_columns:
        print(f" - {col}")
else:
    print("PASS: No obvious label-derived or future-looking columns found.")


# ============================================================
# ATTACK 2: FUTURE-WINDOW ATTACK
# ============================================================

print("\n" + "=" * 70)
print("2. FUTURE-WINDOW ATTACK")
print("=" * 70)

# Check whether month column exists
if "month" in df_leakage.columns:

    df_leakage["month"] = pd.to_datetime(
        df_leakage["month"],
        errors="coerce"
    )

    print("Month range in current dataset:")
    print(f"Minimum month: {df_leakage['month'].min()}")
    print(f"Maximum month: {df_leakage['month'].max()}")

    # Check for obviously future-looking column names
    future_columns = []

    for col in df_leakage.columns:
        col_lower = col.lower()

        if any(word in col_lower for word in [
            "future",
            "next",
            "lead",
            "forward"
        ]):
            future_columns.append(col)

    if future_columns:
        print("\nPotential future-looking columns:")
        for col in future_columns:
            print(f" - {col}")

        print("\nWARNING: These columns require manual inspection.")
    else:
        print("PASS: No explicit future/next/lead columns found.")

    print(
        "\nNote: A complete rolling-window leakage test will be performed "
        "after the rolling-window dataset is created."
    )
    print(
        "At the current stage, we can only audit the existing dataset "
        "for obvious future-looking columns."
    )

else:
    print("WARNING: 'month' column not found.")


# ============================================================
# ATTACK 3: ZERO-VARIANCE / CONSTANT FEATURE ATTACK
# ============================================================

print("\n" + "=" * 70)
print("3. ZERO-VARIANCE / CONSTANT FEATURE ATTACK")
print("=" * 70)

# Only numeric columns
numeric_columns = df_leakage.select_dtypes(
    include=np.number
).columns

constant_columns = []

for col in numeric_columns:

    variance = df_leakage[col].var()

    if variance == 0 or df_leakage[col].nunique(dropna=False) <= 1:
        constant_columns.append(col)

if constant_columns:

    print("Constant / zero-variance columns found:")

    for col in constant_columns:
        print(
            f" - {col} | "
            f"unique values = {df_leakage[col].nunique(dropna=False)}"
        )

else:
    print("PASS: No zero-variance numeric features found.")


# ============================================================
# FINAL LEAKAGE SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("LEAKAGE HUNT SUMMARY")
print("=" * 70)

print(f"Suspicious label/future columns : {len(suspicious_columns)}")
print(f"Constant numeric columns        : {len(constant_columns)}")

print("\nIMPORTANT:")
print("Future-window leakage cannot be fully confirmed until")
print("the rolling-window training table is created.")

print("=" * 70)

LEAKAGE HUNT
Rows    : 2,871,202
Columns : 20

1. LABEL-DERIVED / FUTURE-LOOKING COLUMN ATTACK
PASS: No obvious label-derived or future-looking columns found.

2. FUTURE-WINDOW ATTACK
Month range in current dataset:
Minimum month: 2025-01-01 00:00:00
Maximum month: 2026-06-01 00:00:00
PASS: No explicit future/next/lead columns found.

Note: A complete rolling-window leakage test will be performed after the rolling-window dataset is created.
At the current stage, we can only audit the existing dataset for obvious future-looking columns.

3. ZERO-VARIANCE / CONSTANT FEATURE ATTACK
Constant / zero-variance columns found:
 - ai_other | unique values = 1

LEAKAGE HUNT SUMMARY
Suspicious label/future columns : 0
Constant numeric columns        : 1

IMPORTANT:
Future-window leakage cannot be fully confirmed until
the rolling-window training table is created.


**Remove ai_other**

In [31]:
# ============================================================
# REMOVE ZERO-VARIANCE FEATURE
# ============================================================

df_features_clean = df_features.copy()

# Remove constant feature
if "ai_other" in df_features_clean.columns:
    df_features_clean = df_features_clean.drop(columns=["ai_other"])

print("=" * 70)
print("ZERO-VARIANCE FEATURE REMOVAL")
print("=" * 70)

print("Removed column: ai_other")
print(f"Old shape: {df_features.shape}")
print(f"New shape: {df_features_clean.shape}")

print("\nRemaining columns:")
for i, col in enumerate(df_features_clean.columns, 1):
    print(f"{i:02d}. {col}")

ZERO-VARIANCE FEATURE REMOVAL
Removed column: ai_other
Old shape: (2871202, 20)
New shape: (2871202, 19)

Remaining columns:
01. client_hash_id
02. content_hash_id
03. month
04. gsc_clicks
05. gsc_impressions
06. gsc_avg_position
07. ga4_total_engagement_sec
08. sessions_organic
09. sessions_ai
10. missing_count
11. gsc_avg_position_missing
12. ga4_total_engagement_sec_missing
13. sessions_organic_missing
14. sessions_ai_missing
15. ai_other_missing
16. ctr
17. sec_per_click
18. ai_share
19. engagement_per_organic_session


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 4. What I Excluded and Why

| Excluded Field / Type | Reason for Exclusion |
|:---|:---|
| `ai_other` | Zero-variance feature; contains only 0 values, so it provides no predictive information. |
| **Other Raw Columns** *(from 31-column dataset)* | Redundant, irrelevant to the current prediction objective, or not required for the core feature set. |
| **Unused Metadata / Technical Fields** | Identify or describe the record rather than providing meaningful predictive signals. |
| **Categorical Identifiers** *(client_hash_id, content_hash_id)* | Preserved strictly for tracking, grouping, and split logic; not treated as predictive features. |
| **Future-looking Fields** | Features must be available at prediction time; using future data causes Data Leakage. |
| **Target / Label Fields** *(trend_direction, trend_pct)* | Derived later from future 90-day outcomes; strictly excluded from feature set to prevent leakage. |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.